# Valhalla Routing Workflow for Citi Bike Heat Exposure Analysis

This notebook documents the workflow used to prepare a NYC street network for local Valhalla routing and generate bicycle routes between Citi Bike station pairs. The routed paths can later be intersected with heat exposure or UTCI layers to estimate trip-level thermal exposure.

The workflow includes:
1. Inspecting and tagging the cleaned NYC street network.
2. Converting the street network into Valhalla-compatible routing input.
3. Building and testing a local Valhalla routing server.
4. Loading Citi Bike trips and extracting station-to-station pairs.
5. Routing station pairs through Valhalla.
6. Saving route geometries for later spatial heat exposure analysis.

## 1. Inspect street-network attributes

This section compares the older and newer street-network shapefiles to verify which fields are available and whether the `type` column can be used to assign OSM-style `highway` tags for Valhalla routing.

In [1]:
import geopandas as gpd

# Paths to your shapefiles
old_shp = r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA\SEMESTER\SPRING2025\biketrip-heat-exposure\nyc_streets_neat.shp"
new_shp = r"C:\Users\Agustin\Documents\2025summer\neatnet_generator\verified_streets_neat_bidirectional.shp"

for label, shp in [("OLD", old_shp), ("NEW", new_shp)]:
    gdf = gpd.read_file(shp)
    print(f"\n— {label} SHAPEFILE —")
    # 1) Field names
    print("Fields:", list(gdf.columns))
    # 2) If there’s a ‘type’ column, list its unique values (limit to first 50)
    if 'type' in gdf.columns:
        vals = sorted(gdf['type'].dropna().unique())
        print(f"Unique ‘type’ values ({len(vals)}):", vals[:50], "…")
    else:
        print("No ‘type’ column found.")



— OLD SHAPEFILE —
Fields: ['_status', 'osm_id', 'name', 'ref', 'type', 'oneway', 'bridge', 'maxspeed', 'Shape_Leng', 'geometry']
Unique ‘type’ values (18): ['bridleway', 'busway', 'construction', 'cycleway', 'living_street', 'path', 'platform', 'primary', 'primary_link', 'proposed', 'residential', 'secondary', 'secondary_link', 'tertiary', 'tertiary_link', 'track', 'trunk_link', 'unclassified'] …

— NEW SHAPEFILE —
Fields: ['_status', 'osm_id', 'name', 'ref', 'type', 'oneway', 'bridge', 'maxspeed', 'Shape_Leng', 'geometry']
Unique ‘type’ values (17): ['bridleway', 'construction', 'cycleway', 'footway', 'living_street', 'path', 'pedestrian', 'primary', 'primary_link', 'proposed', 'residential', 'secondary', 'secondary_link', 'tertiary', 'tertiary_link', 'track', 'unclassified'] …


## 2. Tag street network and export GeoJSON

Valhalla expects OSM-style routing attributes. This section loads the cleaned NYC street network, reprojects it to WGS84, assigns routing tags such as `highway`, `bicycle`, `access`, and `oneway`, and exports the result as a GeoJSON file.

In [1]:
import geopandas as gpd
import numpy as np
from pathlib import Path

# 1) Load & reproject
gdf = (
    gpd.read_file(
        r"C:\Users\Agustin\Documents\2025summer\neatnet_generator\verified_streets_neat_bidirectional.shp"
    )
    .to_crs(4326)
)

# 2) Apply your OSM-style tags
fix = {
    'services': 'service',
    'busway':   'service',
    'raceway':  'service',
    'platform': 'footway',
    'planned':  'proposed'
}
gdf['highway'] = gdf['type'].replace(fix).fillna('residential')
gdf['bicycle'] = 'yes'
gdf['access']  = 'yes'

# Your original “oneway_flag” logic
if 'oneway_flag' in gdf.columns:
    gdf['oneway'] = np.where(gdf['oneway_flag'] == 1, 'yes', 'no')
else:
    # If missing, default to “no” to mirror previous behavior
    gdf['oneway'] = 'no'

# 3) Save tagged GeoJSON exactly as before
out = Path(r"C:\valhalla_neat\nyc_streets_neat_tagged.geojson")
gdf[['highway','bicycle','access','oneway','name','geometry']] \
   .to_file(out, driver="GeoJSON")

print(f"Tagged GeoJSON written to {out}")


Tagged GeoJSON written to C:\valhalla_neat\nyc_streets_neat_tagged.geojson


Output: `nyc_streets_neat_tagged.geojson`, saved locally in the Valhalla working directory.

## 3. Convert GeoJSON to OSM and PBF formats

This section converts the tagged GeoJSON file into OSM XML and then into a sorted PBF file. The PBF file is the input used by Valhalla to build routing tiles.

%%cmd
cd C:\valhalla_neat
set WIN=%CD:\=/%
where ogr2osm
ogr2osm --never-upload --encoding=utf-8 -f nyc_streets_neat_tagged.geojson -o verified_nyc_tagged.osm

OSM XML → sorted PBF (two osmconvert calls)

docker run --rm -v %WIN%:/data danieljh/osmium:latest ^
    renumber /data/verified_nyc_tagged.osm -o /data/verified_nyc_sorted.osm --overwrite

docker run --rm -v %WIN%:/data danieljh/osmium:latest ^
    cat /data/verified_nyc_sorted.osm -o /data/verified_nyc_tagged.pbf -f pbf --overwrite

## 4. Build Valhalla tiles and configuration

This section rebuilds the local Valhalla routing tiles from the prepared PBF file and creates the Valhalla configuration file.

rmdir /s /q tiles
mkdir tiles

docker run --rm -v %WIN%:/data --entrypoint /usr/local/bin/valhalla_build_config ^
    ghcr.io/nilsnolde/docker-valhalla/valhalla:latest ^
    --mjolnir-tile-dir /data/tiles ^
    --mjolnir-tile-extract /data/tiles/tiles.tar > valhalla.json

docker run --rm -v %WIN%:/data --entrypoint /usr/local/bin/valhalla_build_tiles ^
    ghcr.io/nilsnolde/docker-valhalla/valhalla:latest ^
    -c /data/valhalla.json /data/verified_nyc_tagged.pbf

## 5. Launch local Valhalla routing service

This section starts the local Valhalla Docker container. The routing service is exposed at `http://localhost:8002`.

```cmd
docker rm -f valhalla_neat 2>nul
docker run -d --name valhalla_neat -p 8002:8002 -v C:/valhalla_neat:/custom_files ghcr.io/nilsnolde/docker-valhalla/valhalla:latest

## 6. Test local Valhalla routing endpoint

This quick test sends a sample bicycle routing request to the local Valhalla server and checks whether a valid response is returned.

In [1]:
import requests, json, urllib.parse as ul

body = {
    "locations": [
        {"lat": 40.758, "lon": -73.985},
        {"lat": 40.742, "lon": -73.971}
    ],
    "costing": "bicycle"
}

url = f"http://localhost:8002/route?json={ul.quote(json.dumps(body))}"
resp = requests.get(url, timeout=5)
print(resp.status_code, resp.json().keys())


200 dict_keys(['trip'])


### At this point, the local routing server is ready and can be used to generate station-to-station bicycle routes.

# END OF VALHALLA ROUTING SETUP AND FILE CREATION

# START OF SHORTEST PATHS CREATION

## 7. Load Citi Bike trips and create station-pair list

This section loads Citi Bike trip records from monthly ZIP files, extracts start and end station information, removes incomplete records, and creates a unique list of station-to-station pairs to route through Valhalla.

In [6]:
# ─── A-FULL · DATA PREP (ALL PAIRS) ─────────────────────────────────
import pandas as pd
import zipfile
import glob
from pathlib import Path

# Make sure these are correct:
WORK     = Path(r"C:\valhalla_neat")
TRIP_DIR = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data"
)

# Only read these columns from each CSV part
USE = [
    "start_station_id", "end_station_id",
    "start_lat", "start_lng",
    "end_lat", "end_lng"
]

def read_one(zip_path: Path) -> pd.DataFrame:
    """
    Load all CSV parts inside a single monthly ZIP (e.g. 202406-citibike-tripdata.zip).
    Previously you were reading just the first CSV; now we concatenate every CSV in the zip.
    """
    frames = []
    with zipfile.ZipFile(zip_path) as z:
        # List every *.csv inside this ZIP
        csv_parts = [f for f in z.namelist() if f.endswith(".csv")]
        for csv_fname in csv_parts:
            df_part = pd.read_csv(
                z.open(csv_fname),
                usecols=USE,
                low_memory=False
            )
            frames.append(df_part)
    return pd.concat(frames, ignore_index=True)

# ——— Iterate over all .zip files in TRIP_DIR (June 2024 only) ———
all_zips = glob.glob(str(TRIP_DIR / "*.zip"))
print(f"► reading {len(all_zips)} ZIP file(s) from {TRIP_DIR} ...")
df_list = [read_one(Path(p)) for p in all_zips]
df = pd.concat(df_list, ignore_index=True)
print(f"► loaded {len(df):,} total rows from all ZIPs")

# ——— Clean station IDs (strip whitespace, cast to string, replace blanks with NA) ———
for col in ["start_station_id", "end_station_id"]:
    df[col] = df[col].astype(str).str.strip().replace({"": pd.NA})

# ——— Build canonical station coordinates (median lat/lon per station ID) ———
xy = (
    df[["start_station_id", "start_lat", "start_lng"]]
    .rename(columns={"start_station_id": "id", "start_lat": "lat", "start_lng": "lon"})
    .groupby("id")
    .median()
)

# ——— Compute the full (start, end) pair list ———
pairs = (
    df[["start_station_id", "end_station_id"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"► unique station pairs: {len(pairs):,}")


► reading 1 ZIP file(s) from C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data ...
► loaded 4,783,576 total rows from all ZIPs
► unique station pairs: 739,187


## 8. Create station point layer

This section converts Citi Bike station coordinates into a GeoDataFrame and reprojects them to EPSG:32118, a projected coordinate system suitable for distance-based snapping and spatial joins in New York City.

In [7]:
# ─── B-FULL · STATION POINTS ───────────────────────────────────────
import geopandas as gpd

# xy: index=station_id (string), columns=[lat, lon]
stations_df = xy.reset_index().rename(columns={"id": "station_id"})
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(
        stations_df.lon,
        stations_df.lat,
        crs="EPSG:4326"
    )
).to_crs(epsg=32118)

print(f"► {len(stations_gdf):,} stations ready (EPSG:32118)")


► 2,274 stations ready (EPSG:32118)


## 9. Configure Valhalla client and route-decoding helpers

This section connects Python to the local Valhalla server and defines helper functions for decoding Valhalla route geometries into Shapely LineStrings.

In [8]:
# ─── C-FULL · CLIENT & HELPERS ─────────────────────────────────────
import sys
# Ensure that Python can find libs/valhalla_routing_client.py
sys.path.append(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure"
)

from libs.valhalla_routing_client import (
    ValhallaBatchRoutingSettings, ValhallaRoutingClient
)
import polyline
import shapely.geometry as sgeom
from math import isfinite

# URL of your locally running Valhalla container
VALHALLA = "http://localhost:8002"
client   = ValhallaRoutingClient(VALHALLA, num_workers=8)

# Routing batch parameters
CHUNK  = 1_000    # number of station‐pairs to send per batch
SNAP_M = 50       # snap tolerance in meters

def decode_poly(shape_str: str):
    """
    Attempt to decode an encoded polyline to a list of (lon, lat) pairs.
    Try both precision=5 and precision=6.
    """
    for prec in (5, 6):
        try:
            pts = polyline.decode(shape_str, precision=prec)
            if pts and all(-90 <= lat <= 90 and -180 <= lon <= 180 for lat, lon in pts):
                return [(lon, lat) for lat, lon in pts]
        except Exception:
            pass
    return None


The notebook relies on the custom helper module `libs/valhalla_routing_client.py`, which should also be included in the GitHub repository if this notebook is uploaded.

## 10. Route station pairs and save raw route geometries

This section sends station-pair coordinates to the local Valhalla server in batches, decodes the returned route geometries, and saves the routed paths as a GeoPackage. Failed routes are logged separately for inspection.

In [10]:
# ─── D-FULL · ROUTE & WRITE RAW OUTPUT (WITH RETRY LOGIC) ─────────────────────────
import geopandas as gpd
import shapely.geometry as sgeom
import contextlib
import io
import pandas as pd
import time
from requests.exceptions import ConnectionError

# ── OUTPUT PATHS ────────────────────────────────────────────────────────────
OUTFILE    = WORK / "verified_station_pair_routes.gpkg"   # updated name
ERRORS_CSV = WORK / "valhalla_errors.csv"

features   = []
bad_pairs  = []  # Will store dicts: {"start_id", "end_id", "error"}
skip_err   = 0
skip_empty = 0
skip_inf   = 0

total_pairs     = len(pairs)
status_interval = 50_000   # print a status line every 50 000 pairs
next_status     = status_interval
processed_pairs = 0

# Reduce parallelism: use a single worker to avoid socket conflicts
client.num_workers = 1

for i in range(0, total_pairs, CHUNK):
    sub = pairs.iloc[i : i + CHUNK]

    coords = []
    meta   = []
    for sid, eid in sub.itertuples(index=False):
        if (sid in xy.index) and (eid in xy.index):
            s_lat, s_lon = xy.loc[sid][["lat", "lon"]]
            e_lat, e_lon = xy.loc[eid][["lat", "lon"]]
            coords.append((s_lat, s_lon, e_lat, e_lon))
            meta.append((sid, eid))
        else:
            bad_pairs.append({
                "start_id": sid,
                "end_id":   eid,
                "error":    "station ID not in xy"
            })
            skip_err += 1

    if coords:
        settings = ValhallaBatchRoutingSettings(
            coords,
            costing="bicycle",
            directions_type="none"
        )
        try:
            settings.search_radius = SNAP_M
        except Exception:
            pass

        # Attempt with retries on connection errors
        attempt, max_retries = 0, 3
        while attempt < max_retries:
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    res = client.get_batch_route(settings)
                break
            except ConnectionError:
                attempt += 1
                time.sleep(2)  # wait 2 seconds before retry
                if attempt == max_retries:
                    # Log entire batch as errors if still failing
                    for sid, eid in meta:
                        bad_pairs.append({
                            "start_id": sid,
                            "end_id":   eid,
                            "error":    "connection error after retries"
                        })
                        skip_err += 1
                    res = [None] * len(meta)

        # Process returned JSONs
        for idx, js in enumerate(res):
            sid, eid = meta[idx]
            if not isinstance(js, dict) or js.get("error"):
                err_msg = js.get("error") if isinstance(js, dict) else "invalid response"
                bad_pairs.append({
                    "start_id": sid,
                    "end_id":   eid,
                    "error":    err_msg
                })
                skip_err += 1
                continue

            shape = (
                js.get("routes", [{}])[0].get("geometry")
                or js.get("trip", {}).get("legs", [{}])[0].get("shape", "")
            )
            path = decode_poly(shape)
            if path is None:
                bad_pairs.append({
                    "start_id": sid,
                    "end_id":   eid,
                    "error":    "decode_poly returned None"
                })
                skip_empty += 1
                continue

            line = sgeom.LineString(path)
            if not all(isfinite(c) for coord in line.coords for c in coord):
                bad_pairs.append({
                    "start_id": sid,
                    "end_id":   eid,
                    "error":    "nonfinite coordinates"
                })
                skip_inf += 1
                continue

            features.append({"pair_id": f"{sid}→{eid}", "geometry": line})

        # Small pause between batches to avoid overwhelming Valhalla
        time.sleep(0.1)

    processed_pairs += len(sub)
    if processed_pairs >= next_status:
        print(f"► Processed {processed_pairs:,} / {total_pairs:,} station-pairs …")
        next_status += status_interval

if processed_pairs < total_pairs:
    print(f"► Processed {processed_pairs:,} / {total_pairs:,} station-pairs (final chunk)")

# Write out all successfully decoded routes (EPSG:4326 → reproject to 32118)
if features:
    gpd.GeoDataFrame(features, crs="EPSG:4326") \
       .to_crs(32118) \
       .to_file(OUTFILE, driver="GPKG")
    print(f"► Wrote {len(features):,} routes to:", OUTFILE)
else:
    print("► No routes were written.")

# Convert bad_pairs list into a DataFrame and save to CSV
if bad_pairs:
    pd.DataFrame(bad_pairs).to_csv(ERRORS_CSV, index=False)
    print(f"► Logged {len(bad_pairs):,} failing pairs to:", ERRORS_CSV)
else:
    print("► No valhalla_errors recorded.")

print(
    f"\nSummary:\n"
    f"  routes added    = {len(features):,}\n"
    f"  valhalla_errors = {skip_err:,}\n"
    f"  empty_shapes    = {skip_empty:,}\n"
    f"  nonfinite       = {skip_inf:,}\n"
)


► Processed 50,000 / 739,187 station-pairs …
► Processed 100,000 / 739,187 station-pairs …
► Processed 150,000 / 739,187 station-pairs …
► Processed 200,000 / 739,187 station-pairs …
► Processed 250,000 / 739,187 station-pairs …
► Processed 300,000 / 739,187 station-pairs …
► Processed 350,000 / 739,187 station-pairs …
► Processed 400,000 / 739,187 station-pairs …
► Processed 450,000 / 739,187 station-pairs …
► Processed 500,000 / 739,187 station-pairs …
► Processed 550,000 / 739,187 station-pairs …
► Processed 600,000 / 739,187 station-pairs …
► Processed 650,000 / 739,187 station-pairs …
► Processed 700,000 / 739,187 station-pairs …
► Wrote 738,979 routes to: C:\valhalla_neat\verified_station_pair_routes.gpkg
► Logged 208 failing pairs to: C:\valhalla_neat\valhalla_errors.csv

Summary:
  routes added    = 738,979
  valhalla_errors = 208
  empty_shapes    = 0
  nonfinite       = 0



## 11. Validate and correct route endpoint station IDs

This section checks whether the start and end points of each routed geometry match the expected Citi Bike stations. It snaps route endpoints to the nearest station within a distance tolerance and preserves the original station-pair IDs for validation.

In [11]:
# ─── E-FULL · RE-ASSIGN STATION IDs (PRESERVE ORIGINAL IDS) ────────────────────────
import geopandas as gpd
from shapely.geometry import Point

# 1) Paths & layer names
RAW_GPKG       = WORK / "verified_station_pair_routes.gpkg"
RAW_LAYER      = "verified_station_pair_routes"
FIXED_GPKG     = WORK / "verified_station_pair_routes_fixed.gpkg"
FIXED_LAYER    = "routes_with_correct_ids"

# 2) Read in the “raw” routes (in EPSG:32118) and add orig_start/orig_end columns
routes = (
    gpd.read_file(RAW_GPKG, layer=RAW_LAYER)
       .reset_index(drop=True)
)

# 3) Recover original start/end IDs from the pair_id string
routes[["orig_start","orig_end"]] = (
    routes["pair_id"]
      .str.split("→", expand=True)
)

# 4) Extract the first and last vertex from each LineString
routes["geom_start"] = routes.geometry.apply(lambda g: Point(g.coords[0]))
routes["geom_end"]   = routes.geometry.apply(lambda g: Point(g.coords[-1]))

tol = 100  # snapping tolerance in metres

def nearest_one(left_gdf, pts_gdf, dist_col, out_col):
    tmp = gpd.sjoin_nearest(
        left_gdf,
        pts_gdf[["station_id", "geometry"]],
        how="left",
        max_distance=tol,
        distance_col=dist_col
    )
    tmp = (
        tmp.reset_index()
           .rename(columns={"index": "idx"})
           .sort_values(["idx", dist_col])
           .drop_duplicates("idx")
           .set_index("idx")[["station_id"]]
           .rename(columns={"station_id": out_col})
    )
    return tmp

# 5) Snap origins and destinations to nearest station_id
routes = routes.join(
    nearest_one(
        routes.set_geometry("geom_start"),
        stations_gdf,
        "start_dist",
        "snapped_start"
    )
)
routes = routes.join(
    nearest_one(
        routes.set_geometry("geom_end"),
        stations_gdf,
        "end_dist",
        "snapped_end"
    )
)

# 6) Count routes that failed to snap
bad = routes[(routes.snapped_start.isna()) | (routes.snapped_end.isna())]
print(f"routes without snap ≤ {tol} m: {len(bad):,}")

# 7) Keep only columns needed downstream
fixed = routes[["orig_start","orig_end","geometry"]].copy()
fixed = fixed.set_crs("EPSG:32118")

fixed.to_file(
    FIXED_GPKG,
    layer=FIXED_LAYER,
    driver="GPKG"
)
print("FIXED layer written to:", FIXED_GPKG)


routes without snap ≤ 100 m: 335
FIXED layer written to: C:\valhalla_neat\verified_station_pair_routes_fixed.gpkg


## 12. Optional validation: inspect routes with unmatched endpoints

This optional section identifies routes where the start or end point could not be matched to a station within the snapping tolerance. These cases can be reviewed manually or excluded from later analysis.

In [12]:
# ─── F-FULL · INSPECT REMAINING MIS-SNAPS (OPTIONAL) ───────────────
from shapely.ops import nearest_points
from shapely.geometry import Point
import pandas as pd

# 'routes' is the GeoDataFrame from E-FULL, containing:
#   orig_start, orig_end, snapped_start, snapped_end, geometry

missing = routes[(routes.snapped_start.isna()) | (routes.snapped_end.isna())].copy()
if missing.empty:
    print("✅ All routes snapped within tolerance")
else:
    def dist_to_station(pt: Point):
        nearest = nearest_points(pt, stations_gdf.unary_union)[1]
        return pt.distance(nearest)

    # Compute distances for each mis-snapped line end
    missing["dist_start_m"] = missing.geometry.apply(
        lambda g: dist_to_station(Point(g.coords[0]))
    )
    missing["dist_end_m"] = missing.geometry.apply(
        lambda g: dist_to_station(Point(g.coords[-1]))
    )

    pd.options.display.float_format = "{:.1f}".format
    display_cols = ["orig_start", "orig_end", "dist_start_m", "dist_end_m"]
    print(missing[display_cols])


C:\Users\Agustin\AppData\Local\Temp\ipykernel_52600\4097265011.py:14: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  nearest = nearest_points(pt, stations_gdf.unary_union)[1]
C:\Users\Agustin\AppData\Local\Temp\ipykernel_52600\4097265011.py:14: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  nearest = nearest_points(pt, stations_gdf.unary_union)[1]


       orig_start orig_end  dist_start_m  dist_end_m
3757      4440.02  6517.08         278.0        10.2
7634      4440.02  4546.06         278.0         3.3
16736     4374.01  5128.04         246.1         7.5
17570     4440.02  5128.04         278.0         7.5
36964     3019.03  3202.06         118.5         0.6
...           ...      ...           ...         ...
737351    4458.06   SYS025           4.1       154.7
737554    3318.05   SYS025           2.0       154.7
737577    3958.02   SYS025           3.8       154.7
737641    4519.02   SYS025           7.9       154.7
737766    5763.03   SYS025           5.2       154.7

[335 rows x 4 columns]
